In [1]:
from argparse import ArgumentParser
from datetime import datetime
from torch.utils.data import DataLoader
import logging
import os
from models.lstm_proj_diff import LSTM
from utils.dataset import TranslateDataset
from utils.earlystopper import EarlyStopper
import pandas as pd
from logging import getLogger
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
import json
from utils.warmup import WarmupScheduler

In [2]:
df_train = pd.read_parquet("data/tokenized_train.parquet")

In [3]:
df_train.head()

,en,pt,en_length,pt_length,en_max_len_word,pt_max_len_word,en_tokens,pt_tokens
0,"These people built, in stone, objects which th...","Quando esses objetos se foram, eles começaram ...",158,159,7,13,"[5, 5139, 1134, 4400, 460, 781, 4462, 460, 109...","[5, 1599, 1930, 9621, 766, 1887, 460, 1031, 67..."
1,I've spent my whole life trying to win his aff...,"Passei minha vida tentando ganhar o amor, a ap...",101,95,11,9,"[5, 716, 468, 789, 3721, 891, 2146, 1317, 1938...","[5, 14076, 1105, 1225, 2300, 3979, 411, 2739, ..."
2,I see you manage my family every day with grac...,Vejo você gerenciar minha família todos os dia...,124,126,11,12,"[5, 716, 1259, 778, 11042, 891, 1771, 1540, 11...","[5, 22128, 849, 31947, 1105, 1841, 1221, 741, ..."
3,"Five years ago, my friends moved to london, bu...","Há cinco anos, meus amigos se mudaram para lon...",111,122,12,13,"[5, 21172, 1372, 1851, 460, 891, 2411, 5045, 7...","[5, 3211, 3216, 1165, 460, 1909, 2506, 766, 14..."
4,"Illarion shevardnadze, don't come near me, or ...","Illarion shevardnadze, não se aproxime de mim ...",107,110,13,13,"[5, 44, 79, 1821, 1993, 3985, 4675, 71, 10309,...","[5, 44, 79, 1821, 1993, 3985, 4675, 71, 10309,..."


In [4]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

tokenizer: Tokenizer = Tokenizer.from_file('artifacts/tokenizer_50000.json')

In [5]:
df_train.en.values[0]

'These people built, in stone, objects which they seem to have seen in the sky, in flight, at some point, no doubt, landed on the surface of the earth as well.'

In [10]:
for _, row in df_train.sample(100).iterrows():
    print('Frase inglês:', row.en)
    print('Decoding tokens inglês:', tokenizer.decode(row.en_tokens))
    print('Frase português:', row.pt)
    print('Decoding tokens português:', tokenizer.decode(row.pt_tokens))
    print('---'*80)

Frase inglês: This is our chance to see if we have what it takes, and like grandpa jack says, we're part of a team.
Decoding tokens inglês: This is our chance to see if we have what it takes , and like grandpa jack says , we ' re part of a team .
Frase português: Esta é a nossa chance para ver se nós podemos, e como o vovô jack diz, se somos parte da equipe.
Decoding tokens português: Esta é a nossa chance para ver se nós podemos , e como o vovô jack diz , se somos parte da equipe .
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Frase inglês: Transmit our position to over 300 nigerian soldiers who ran all night to get to within an hour and a half of where we are now?
Decoding tokens inglês: Transmit our position to over 300 nigerian soldiers who ran all night to get to within an hour and a half 

In [6]:
df_train.en_tokens.values[0]

array([    5,  5139,  1134,  4400,   460,   781,  4462,   460, 10969,
        1247,   939,  4087,   737,   897,  2627,   781,   739,  4424,
         460,   781,  6561,   460,   760,  1187,  2374,   460,   857,
        5959,   460, 12067,   769,   739,  6411,   770,   739,  2607,
         736,  1521,   441,     6])

In [8]:
tokenizer.decode(df_train.pt_tokens.values[0])

'Quando esses objetos se foram , eles começaram a reproduzí - los em pedra para que pudessem lembrar que estes eram os instrumentos em que os deuses vieram a eles .'